# Week 7 — Advanced Models (Gradient Boosting)
**Internship:** IDX Exchange Data Science Program  
**Name:** Monika  
**Week:** 6  
**Dataset:** CRMLS Sold Properties, cleaned in Week 3, models from Week 5 & 6

**Goal:** Train an XGBoost model, perform light hyperparameter tuning on 
max_depth, learning_rate, and n_estimators, and compare against Linear 
Regression, Decision Tree, and Random Forest from prior weeks.

Uses the same 3-month training window and the updated (Week 6) feature set, 
including BedBathRatio and the one-hot encoded SchoolDistrict.

In [1]:
!pip install xgboost --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\monik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Setup
Loading the cleaned dataset and reusing the same 3-month training window and 
June 2026 test month as every prior week, so this comparison stays fair.

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_percentage_error

data_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\data\california'
model_df = pd.read_csv(data_folder + '\\cleaned_full.csv', parse_dates=['CloseDate_parsed'])

BASE_FEATURE_COLS = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres',
    'PropertyAge', 'DaysOnMarket', 'Latitude', 'Longitude',
    'PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'AssociationFee',
    'LivingArea_missing', 'BathroomsTotalInteger_missing',
    'YearBuilt_missing', 'LotSizeAcres_missing', 'DaysOnMarket_anomaly'
]
target_col = 'ClosePrice'
BEST_WINDOW = 3
test_month = pd.Period('2026-06', freq='M')

def get_train_test_split(frame, test_month, window_months):
    frame = frame.copy()
    frame['YearMonth'] = frame['CloseDate_parsed'].dt.to_period('M')
    test_df = frame[frame['YearMonth'] == test_month]
    train_start = test_month - window_months
    train_df = frame[(frame['YearMonth'] >= train_start) & (frame['YearMonth'] < test_month)]
    return train_df.drop(columns='YearMonth'), test_df.drop(columns='YearMonth')

print(f'Loaded {len(model_df):,} rows')

Loaded 411,419 rows


## 1. Rebuild Week 6 Features
XGBoost is a separate model from Random Forest, so this notebook rebuilds 
BedBathRatio and the school-district spatial join from scratch rather than 
depending on notebook 5 having been run first — keeps this notebook 
self-contained and runnable on its own.

In [3]:
model_df['BedBathRatio'] = model_df['BedroomsTotal'] / model_df['BathroomsTotalInteger'].replace(0, np.nan)
model_df['BedBathRatio'] = model_df['BedBathRatio'].fillna(model_df['BedBathRatio'].median())

district_path = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\scripts\DistrictAreas2526_-284845464123469011.geojson'
districts = gpd.read_file(district_path)
districts_unified = districts[districts['DistrictType'] == 'Unified'].copy()

properties_gdf = gpd.GeoDataFrame(
    model_df,
    geometry=gpd.points_from_xy(model_df['Longitude'], model_df['Latitude']),
    crs='EPSG:4326'
).to_crs(districts_unified.crs)

joined = gpd.sjoin(properties_gdf, districts_unified[['DistrictName', 'geometry']], how='left', predicate='within')
model_df['SchoolDistrict'] = joined['DistrictName'].values
model_df['SchoolDistrict'] = model_df['SchoolDistrict'].fillna('No_Unified_District')
model_df = model_df[~model_df.index.duplicated(keep='first')]

print(f'Rows after rebuilding features: {len(model_df):,}')

Rows after rebuilding features: 411,419


## 2. Build Train/Test Split with One-Hot Encoded SchoolDistrict
Same setup as Week 6: SchoolDistrict is one-hot encoded with one category 
dropped as reference, then joined back onto the train/test data.

In [4]:
train_df, test_df = get_train_test_split(model_df, test_month, BEST_WINDOW)

train_dummies = pd.get_dummies(train_df['SchoolDistrict'], prefix='Dist', drop_first=True)
test_dummies = pd.get_dummies(test_df['SchoolDistrict'], prefix='Dist', drop_first=True)
train_dummies, test_dummies = train_dummies.align(test_dummies, join='left', axis=1, fill_value=0)

train_df = pd.concat([train_df.reset_index(drop=True), train_dummies.reset_index(drop=True)], axis=1)
test_df = pd.concat([test_df.reset_index(drop=True), test_dummies.reset_index(drop=True)], axis=1)

DISTRICT_COLS = train_dummies.columns.tolist()
FEATURE_COLS = BASE_FEATURE_COLS + ['BedBathRatio'] + DISTRICT_COLS

X_train, y_train = train_df[FEATURE_COLS], train_df[target_col]
X_test, y_test = test_df[FEATURE_COLS], test_df[target_col]

print(f'Train rows: {len(X_train):,} | Test rows: {len(X_test):,} | Features: {len(FEATURE_COLS)}')

Train rows: 35,186 | Test rows: 12,841 | Features: 311


## 3. Light Hyperparameter Tuning
XGBoost has 3 main "dials" that control how it learns:
- **max_depth** — how complex each individual tree is allowed to get
- **learning_rate** — how much each new tree corrects the previous trees' 
  mistakes (lower = more cautious, needs more trees to compensate)
- **n_estimators** — how many trees to build in total

Tested 5 reasonable combinations on a validation split (carved out of training 
data, test set untouched) rather than an exhaustive search — enough to find a 
solid setting without over-engineering this step.

In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

param_grid = [
    {'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 200},
    {'max_depth': 4, 'learning_rate': 0.1,  'n_estimators': 300},
    {'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 300},
    {'max_depth': 6, 'learning_rate': 0.1,  'n_estimators': 200},
    {'max_depth': 8, 'learning_rate': 0.05, 'n_estimators': 300},
]

tuning_results = []
for params in param_grid:
    model = XGBRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    val_r2 = r2_score(y_val, model.predict(X_val))
    tuning_results.append({**params, 'val_R2': val_r2})

tuning_df = pd.DataFrame(tuning_results).sort_values('val_R2', ascending=False)
tuning_df

,max_depth,learning_rate,n_estimators,val_R2
4,8,0.05,300,0.886440
3,6,0.10,200,0.876251
2,6,0.05,300,0.871078
1,4,0.10,300,0.857784
0,4,0.05,200,0.817915


In [6]:
best_params = tuning_df.iloc[0][['max_depth', 'learning_rate', 'n_estimators']].to_dict()
best_params['max_depth'] = int(best_params['max_depth'])
best_params['n_estimators'] = int(best_params['n_estimators'])
print(f'Best params: {best_params}')

Best params: {'max_depth': 8, 'learning_rate': 0.05, 'n_estimators': 300}


## 4. Train Final XGBoost Model and Evaluate on Test Set
Retraining on the full training set (not just the validation split) using the 
winning combination: max_depth=8, learning_rate=0.05, n_estimators=300.

In [7]:
xgb_model = XGBRegressor(**best_params, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

xgb_train_r2 = r2_score(y_train, xgb_model.predict(X_train))
xgb_test_r2 = r2_score(y_test, xgb_model.predict(X_test))
xgb_test_mape = mean_absolute_percentage_error(y_test, xgb_model.predict(X_test))

print(f'XGBoost — Train R²: {xgb_train_r2:.4f} | Test R²: {xgb_test_r2:.4f} | Test MAPE: {xgb_test_mape:.4f}')

XGBoost — Train R²: 0.9410 | Test R²: 0.8925 | Test MAPE: 0.1536


## 5. Compare Against All Prior Models

In [8]:
final_comparison = pd.DataFrame([
    {'model': 'Linear Regression', 'test_R2': 0.649220},
    {'model': 'Decision Tree',     'test_R2': 0.774022},
    {'model': 'Random Forest',     'test_R2': 0.875719},
    {'model': 'XGBoost',           'test_R2': xgb_test_r2},
])
final_comparison['overfit_gap'] = [np.nan, np.nan, np.nan, xgb_train_r2 - xgb_test_r2]
final_comparison

,model,test_R2,overfit_gap
0,Linear Regression,0.649220,NaN
1,Decision Tree,0.774022,NaN
2,Random Forest,0.875719,NaN
3,XGBoost,0.892493,0.048542


## 6. Feature Importance
Which features XGBoost relied on most to make predictions.

In [9]:
xgb_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'xgb_importance': xgb_model.feature_importances_
}).sort_values('xgb_importance', ascending=False)

xgb_importance.head(15)

,feature,xgb_importance
2,BathroomsTotalInteger,0.130504
186,Dist_Newport-Mesa Unified,0.032904
201,Dist_Palo Alto Unified,0.032679
6,Latitude,0.030555
157,Dist_Manhattan Beach Unified,0.028291
7,Longitude,0.027850
0,LivingArea,0.025412
253,Dist_Santa Monica-Malibu Unified,0.024730
187,Dist_No_Unified_District,0.022306
133,Dist_Laguna Beach Unified,0.015845


## Summary of Findings

**Hyperparameter tuning:** Tested 5 combinations of max_depth, learning_rate, 
and n_estimators on a validation split. Validation R² ranged from 0.818 (low 
depth, few trees) up to 0.886 (max_depth=8, learning_rate=0.05, 
n_estimators=300), showing that both tree depth and number of trees mattered 
-- shallow/few-tree combinations clearly underfit relative to the deeper, 
larger ensemble.

**Final results:**

| Model              | Test R² | Overfit Gap |
|--------------------|---------|-------------|
| Linear Regression  | 0.649   | --          |
| Decision Tree      | 0.774   | --          |
| Random Forest      | 0.876   | --          |
| **XGBoost**        | **0.892** | 0.049     |

XGBoost is the best-performing model so far, improving test R² by 0.017 over 
Random Forest and cutting error further (MAPE = 15.4%, comparable to Random 
Forest's 15.2% -- essentially tied on typical error size, but XGBoost explains 
more of the overall price variance). Its overfit gap (0.049, train R² 0.941 
vs test R² 0.892) is reasonable and in a similar range to Random Forest and 
Decision Tree, not a red flag of memorization.

**Feature importance:** BathroomsTotalInteger remains the single most 
important feature (13.1%), consistent with every prior week. What's new here: 
several *individual* school districts now rank among the top 15 most 
important features on their own -- Newport-Mesa Unified, Palo Alto Unified, 
Manhattan Beach Unified, Santa Monica-Malibu Unified, and Laguna Beach Unified 
all appear high on the list. These are well-known high-value/high-demand CA 
markets, so it makes sense the model is learning that being in one of these 
specific districts carries real, independent pricing signal beyond just 
raw Latitude/Longitude. This is a more granular and interpretable version of 
the geographic signal than Week 6 showed with Random Forest, where the 
district feature barely moved the needle -- XGBoost's sequential 
error-correcting approach appears better able to isolate which *specific* 
districts matter most, rather than treating location as one blended signal.

**Best model overall: XGBoost (test R² = 0.892, MAPE = 15.4%)** -- now the 
model to beat heading into Week 8's expanded evaluation.